# Notebook 4 (v2): Cross-Source Test — Cep Telefonu Fotoğrafları

Bu notebook egitilmis DINOv2 modelini Google Street View disinda cekilen gercek cep telefonu fotograflariyla test eder.

## v2.2 degisiklikleri

- v5 ana model artifact'lerine yonlendirildi: ViT-B/14 + CLS descriptor.
- Drive'dan toplu yukleme: `files.upload()` yerine Drive'daki bir klasorden okur.
- Hucre 7 gorsellestirmesi Top-1/Top-2/Top-3 filepath'lerini dogru kullanir.
- Unicode sokak adi normalizasyonu, EXIF Orientation duzeltmesi, telefon fotograflari icin oran koruyan sorgu kirpma ve LightGlue `rbd()` match sayimi korunur.
- LightGlue yardimci tani metrigi olarak raporlanir; ana karar DINOv2 Top-1/Top-K aday kalitesidir.

## Kullanim

1. v5 ana egitim notebook'unu calistir ve artifact'leri Drive'a uret.
2. Cep telefonu fotograflarini `My Drive/Kirsehir_VPR_CrossSource_Photos/` klasorune koy.
3. Bu notebook'u bastan sona calistir.

## On kosullar
- `dinov2_kirsehir_v5_vitb14_cls_domainaug_best.pth`
- `faiss_index_v5_all.bin`
- `db_meta_v5_all.json`

In [ ]:
# ============================================================
# Hücre 1: Kütüphane Kurulumu
# ============================================================
!pip install faiss-cpu folium Pillow pillow-heif -q
!pip install git+https://github.com/cvg/LightGlue.git -q
print("✅ Kütüphaneler kuruldu (HEIC desteği dahil).")

In [ ]:
# ============================================================
# Hücre 2: Konfigürasyon ve Model/Veritabanı Yükleme
# ============================================================
import os, json, random, math, time, glob, shutil, zipfile, unicodedata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 130
from PIL import Image, ImageOps
from PIL.ExifTags import TAGS, GPSTAGS

# HEIC desteği için pillow-heif'i PIL'e kaydet
try:
    from pillow_heif import register_heif_opener
    register_heif_opener()
    print("HEIC desteği aktif (iPhone fotoğrafları okunabilir).")
except ImportError:
    print("pillow-heif kurulu değil; HEIC dosyaları okunamayacak.")

from tqdm import tqdm
from collections import Counter, defaultdict
from IPython.display import display, HTML

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import faiss
import folium

from lightglue import LightGlue, ALIKED
from lightglue.utils import load_image as _lg_load_image_orig, rbd
import torch as _torch_for_heic

def open_rgb_image(path):
    """PIL görselini EXIF Orientation düzeltilmiş RGB olarak açar."""
    img = Image.open(path)
    img = ImageOps.exif_transpose(img)
    return img.convert("RGB")

LIGHTGLUE_MAX_SIDE = 1024
RESAMPLE_LANCZOS = getattr(Image, "Resampling", Image).LANCZOS

def load_image(path):
    """LightGlue için HEIC/JPEG/PNG görsellerini EXIF yönü düzeltilmiş tensor olarak açar."""
    try:
        img_pil = open_rgb_image(path)
        if max(img_pil.size) > LIGHTGLUE_MAX_SIDE:
            img_pil.thumbnail((LIGHTGLUE_MAX_SIDE, LIGHTGLUE_MAX_SIDE), RESAMPLE_LANCZOS)
        img_np = np.array(img_pil)
        return _torch_for_heic.from_numpy(img_np).permute(2, 0, 1).float() / 255.0
    except Exception:
        return _lg_load_image_orig(path)

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Cihaz: {DEVICE}")

# ─── Drive bağla ───
from google.colab import drive
drive.mount("/content/drive")

# ─── KONFİGÜRASYON ──────────────────────────────────────────
DRIVE_DIR   = "/content/drive/My Drive/Kirsehir_VPR_DINOv2_v5_Final"
RESULTS_DIR = "/content/drive/My Drive/Kirsehir_VPR_Results"
PHOTOS_DIR  = "/content/drive/My Drive/Kirsehir_VPR_CrossSource_Photos"
MANUAL_GPS_CSV = os.path.join(PHOTOS_DIR, "cross_source_gps.csv")
# ────────────────────────────────────────────────────────────

os.makedirs(RESULTS_DIR, exist_ok=True)

MODEL_NAME = "dinov2_vitb14"
POOLING = "cls"
MODEL_PATH = os.path.join(DRIVE_DIR, "dinov2_kirsehir_v5_vitb14_cls_domainaug_best.pth")
FAISS_PATH = os.path.join(DRIVE_DIR, "faiss_index_v5_all.bin")
META_PATH  = os.path.join(DRIVE_DIR, "db_meta_v5_all.json")

DRIVE_ZIP  = "/content/drive/My Drive/kirsehir_data.zip"
LOCAL_ROOT = "/content/kirsehir_data"

OUTFEATURES = 512
IMG_SIZE = 518
RERANK_TOP_K = 20

class GeMPooling(nn.Module):
    def __init__(self, p=3, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.ones(1) * p)
        self.eps = eps

    def forward(self, x):
        return x.clamp(min=self.eps).pow(self.p).mean(dim=1).pow(1.0 / self.p)

class VPRDINOv2(nn.Module):
    def __init__(self, backbone_name=MODEL_NAME, out_dim=OUTFEATURES, pooling=POOLING):
        super().__init__()
        self.pooling = pooling.lower()
        self.backbone = torch.hub.load("facebookresearch/dinov2", backbone_name, verbose=False)
        embed_dim = self.backbone.embed_dim

        if self.pooling == "gem":
            self.gem = GeMPooling(p=3)
        elif self.pooling != "cls":
            raise ValueError(f"Desteklenmeyen pooling: {pooling}")

        self.head = nn.Sequential(
            nn.Linear(embed_dim, embed_dim),
            nn.BatchNorm1d(embed_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.10),
            nn.Linear(embed_dim, out_dim),
        )

    def forward(self, x):
        ret = self.backbone.forward_features(x)
        if self.pooling == "cls":
            x = ret["x_norm_clstoken"]
        else:
            x = self.gem(ret["x_norm_patchtokens"])
        x = self.head(x)
        return nn.functional.normalize(x, p=2, dim=1)

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f"Model bulunamadi. Once v5 egitim notebook'unu calistir:\n{MODEL_PATH}")

model = VPRDINOv2(MODEL_NAME, OUTFEATURES, POOLING).to(DEVICE)
state = torch.load(MODEL_PATH, map_location=DEVICE)
if isinstance(state, dict) and "model_state" in state:
    state = state["model_state"]
model.load_state_dict(state)
model.eval()
print(f"DINOv2 v5 modeli yüklendi: {MODEL_NAME} / {POOLING}")

eval_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE), interpolation=T.InterpolationMode.BICUBIC),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Telefon fotoğrafları çoğu zaman dikey/4:3 gelir. Kareye zorla esnetmek
# perspektifi bozduğu için sorguda oranı koruyup merkezden kare kırpıyoruz.
phone_query_transform = T.Compose([
    T.Resize(IMG_SIZE, interpolation=T.InterpolationMode.BICUBIC),
    T.CenterCrop((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# ─── LightGlue ───
lg_extractor = ALIKED(max_num_keypoints=1024).eval().to(DEVICE)
lg_matcher = LightGlue(features="aliked").eval().to(DEVICE)
print("LightGlue yüklendi.")

# ─── FAISS ───
if os.path.exists(FAISS_PATH) and os.path.exists(META_PATH):
    faiss_index = faiss.read_index(FAISS_PATH)
    with open(META_PATH, "r", encoding="utf-8") as f:
        db_meta = json.load(f)
    print(f"FAISS indeksi yüklendi ({faiss_index.ntotal} vektör)")
else:
    raise FileNotFoundError(
        f"FAISS indeksi veya metadata bulunamadı:\n{FAISS_PATH}\n{META_PATH}\n"
        "Önce v5 ana eğitim notebook'unu çalıştır ve artifact'lerin Drive'da olduğunu doğrula.")

# ─── Veri seti dosyalarına LOCAL_ROOT'tan erişim ───
if not os.path.exists(LOCAL_ROOT) and os.path.exists(DRIVE_ZIP):
    print("Veri seti zip dosyasından açılıyor...")
    shutil.copy2(DRIVE_ZIP, "/content/kirsehir_data.zip")
    with zipfile.ZipFile("/content/kirsehir_data.zip", "r") as zf:
        zf.extractall(LOCAL_ROOT)
    os.remove("/content/kirsehir_data.zip")
    print("Veri seti açıldı.")
elif os.path.exists(LOCAL_ROOT):
    print(f"Veri seti zaten mevcut: {LOCAL_ROOT}")
else:
    print("Veri seti yok; görselleştirme aşaması sınırlı çalışabilir.")

def haversine(lat1, lon1, lat2, lon2):
    R = 6371000
    p1 = math.radians(lat1); p2 = math.radians(lat2)
    dp = math.radians(lat2 - lat1); dl = math.radians(lon2 - lon1)
    a = math.sin(dp / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

print("\nSistem hazır.")

In [ ]:
# ============================================================
# Hücre 3: Cep Telefonu Fotoğraflarını Drive'dan Topla (HEIC + alt klasör)
# ============================================================
# Klasör yapısı:
#   PHOTOS_DIR/
#     ├── 01_545 Sokak/
#     │   ├── IMG_3128.HEIC
#     │   └── ...
#     ├── 02_Obruk Caddesi/
#     │   └── ...
#     └── ...
#
# Klasör adları "01_", "02_" gibi prefix içeriyor ve sokak numaraları
# DB'deki "545. Sokak" formatından farklı ("545 Sokak"). Aşağıdaki eşleme
# tablosu klasör adını DB'deki gerçek sokak adına çevirir.

def get_exif_gps(image_path):
    """EXIF'ten GPS koordinatlarını çıkarır (HEIC için de çalışır)."""
    try:
        img = Image.open(image_path)
        # HEIC için _getexif() farklı çalışır, getexif() daha güvenli
        try:
            exif_data = img._getexif()
        except AttributeError:
            exif_data = img.getexif()
        if exif_data is None or len(exif_data) == 0:
            return None, None
        # Sözlüğe çevir
        if hasattr(exif_data, 'items'):
            exif_dict = dict(exif_data.items())
        else:
            exif_dict = exif_data

        gps_info = {}
        for tag_id, value in exif_dict.items():
            tag = TAGS.get(tag_id, tag_id)
            if tag == "GPSInfo":
                # HEIC'te value bazen IFD pointer olabilir
                if hasattr(exif_data, 'get_ifd'):
                    try:
                        gps_ifd = exif_data.get_ifd(tag_id)
                        for gps_tag_id, gps_val in gps_ifd.items():
                            gps_tag = GPSTAGS.get(gps_tag_id, gps_tag_id)
                            gps_info[gps_tag] = gps_val
                    except Exception:
                        pass
                else:
                    for gps_tag_id in value:
                        gps_tag = GPSTAGS.get(gps_tag_id, gps_tag_id)
                        gps_info[gps_tag] = value[gps_tag_id]
        if not gps_info:
            return None, None
        def to_deg(value):
            d, m, s = value
            return float(d) + float(m)/60 + float(s)/3600
        lat = to_deg(gps_info.get("GPSLatitude", (0,0,0)))
        lon = to_deg(gps_info.get("GPSLongitude", (0,0,0)))
        if gps_info.get("GPSLatitudeRef", "N") == "S":
            lat = -lat
        if gps_info.get("GPSLongitudeRef", "E") == "W":
            lon = -lon
        if lat == 0 and lon == 0:
            return None, None
        return lat, lon
    except Exception as e:
        return None, None


def norm_text(s):
    """Drive/Colab kaynaklı NFC/NFD Unicode farklarını tek biçime indirir."""
    return unicodedata.normalize("NFC", str(s)).strip()


# ─── KLASÖR ADI → DB SOKAK ADI EŞLEME TABLOSU ───
# Soldaki: senin klasör adın
# Sağdaki: db_meta'da geçen tam sokak adı
folder_to_street = {
    "01_545 Sokak":                          "545. Sokak",
    "02_Obruk Caddesi":                      "Obruk Caddesi",
    "03_Mustafa Kemal Paşa Caddesi":         "Mustafa Kemal Paşa Caddesi",
    "04_452 Sokak":                          "452. Sokak",
    "05_418 Sokak":                          "418. Sokak",
    "06_417 Sokak":                          "417. Sokak",
    "07_İkizarası Caddesi":                  "İkizarası Caddesi",
    "11_Atatürk Bulvarı":                    "Atatürk Bulvarı",
    "12_1090 Sokak":                         "1090. Sokak",
    "13_Şehit Sahir Kurutluoğlu Caddesi":    "Şehit Sahir Kurutluoğlu Caddesi",
    "14_1683 Sokak":                         "1683. Sokak",
}
folder_to_street = {norm_text(k): norm_text(v) for k, v in folder_to_street.items()}

# Eğer klasör adı tabloda yoksa otomatik bir "tahmin" üretmeyi denesin diye:
def auto_guess_street(folder_name, db_street_lookup):
    """Klasör adından prefix temizleyip DB'deki gerçek sokak adına çözer."""
    cleaned = norm_text(folder_name).split("_", 1)[-1] if "_" in norm_text(folder_name) else norm_text(folder_name)
    if cleaned in db_street_lookup:
        return db_street_lookup[cleaned]
    # "545 Sokak" → "545. Sokak"
    parts = cleaned.split(" ", 1)
    if len(parts) == 2 and parts[0].isdigit():
        candidate = f"{parts[0]}. {parts[1]}"
        if candidate in db_street_lookup:
            return db_street_lookup[candidate]
    return None


# ─── Drive klasöründeki tüm alt klasörleri tara ───
if not os.path.exists(PHOTOS_DIR):
    raise FileNotFoundError(
        f"Fotoğraf klasörü bulunamadı: {PHOTOS_DIR}\n"
        f"Hücre 2'de PHOTOS_DIR yolunu doğrula.")

# HEIC dahil destekleniyor
supported_ext = (".jpg", ".jpeg", ".png", ".heic", ".heif")

subfolders = sorted([
    f for f in os.listdir(PHOTOS_DIR)
    if os.path.isdir(os.path.join(PHOTOS_DIR, f))
])

print(f"📁 Ana klasör: {PHOTOS_DIR}")
print(f"📂 Alt klasör sayısı: {len(subfolders)}\n")

# Manuel GPS CSV
manual_gps = {}
if os.path.exists(MANUAL_GPS_CSV):
    df_gps = pd.read_csv(MANUAL_GPS_CSV)
    for _, row in df_gps.iterrows():
        manual_gps[row["filename"]] = (float(row["lat"]), float(row["lon"]))
    print(f"✅ Manuel GPS CSV'si bulundu: {len(manual_gps)} kayıt\n")

db_street_lookup = {norm_text(m["street"]): m["street"] for m in db_meta}
db_streets = set(db_street_lookup.keys())

# Her alt klasördeki fotoğrafları tara
cross_source_data = []
unmatched_folders = []
folder_summary = {}

for folder_name in subfolders:
    folder_path = os.path.join(PHOTOS_DIR, folder_name)
    photos_in_folder = sorted([
        f for f in os.listdir(folder_path)
        if f.lower().endswith(supported_ext)
    ])
    if not photos_in_folder:
        continue

    # Klasör → DB sokak adı çözümlemesi
    folder_key = norm_text(folder_name)
    if folder_key in folder_to_street:
        gt_street = folder_to_street[folder_key]
        resolution = "manuel eşleme"
    else:
        guessed = auto_guess_street(folder_name, db_street_lookup)
        if guessed:
            gt_street = guessed
            resolution = "otomatik tahmin"
        else:
            gt_street = folder_name  # Olduğu gibi
            resolution = "EŞLEŞME YOK"

    gt_key = norm_text(gt_street)
    is_matched = gt_key in db_street_lookup
    if is_matched:
        gt_street = db_street_lookup[gt_key]
    if not is_matched:
        unmatched_folders.append((folder_name, gt_street))

    folder_summary[folder_name] = {
        "n_photos": len(photos_in_folder),
        "gt_street": gt_street,
        "resolution": resolution,
        "matched": is_matched,
    }

    for fname in photos_in_folder:
        fpath = os.path.join(folder_path, fname)
        lat, lon = get_exif_gps(fpath)
        gps_source = None
        if lat is not None:
            gps_source = "EXIF"
        elif fname in manual_gps:
            lat, lon = manual_gps[fname]
            gps_source = "manual"

        cross_source_data.append({
            "filename": fname,
            "filepath": fpath,
            "folder_name": folder_name,
            "folder_street": gt_street,  # ← Çözümlenmiş DB sokak adı
            "lat": lat,
            "lon": lon,
            "gps_source": gps_source,
        })

# ─── Klasör özeti ───
print(f"{'='*100}")
print(f"📂 KLASÖR ÖZETİ (klasör adı → DB sokak adı eşlemesi)")
print(f"{'='*100}")
print(f"{'Klasör':<40} {'→ DB Sokak Adı':<35} {'Foto':>5} {'Eşleşme':>10}")
print(f"{'─'*100}")
for fname, info in folder_summary.items():
    match = "✅" if info["matched"] else "❌"
    print(f"{fname[:38]:<40} → {info['gt_street'][:33]:<35} "
          f"{info['n_photos']:>5} {match:>10}")

if unmatched_folders:
    print(f"\n⚠️ DB'de OLMAYAN sokaklara çözümlenen klasörler:")
    from difflib import get_close_matches
    for folder, attempted in unmatched_folders:
        suggestions = get_close_matches(norm_text(attempted), db_streets, n=3, cutoff=0.5)
        sugg_str = f" → öneriler: {suggestions}" if suggestions else ""
        print(f"   - '{folder}' → '{attempted}' DB'de yok!{sugg_str}")
    print(f"\n   Yukarıdaki Hücre 3 başında 'folder_to_street' sözlüğünü güncelle.")

# Genel özet
n_exif = sum(1 for d in cross_source_data if d["gps_source"] == "EXIF")
n_manual = sum(1 for d in cross_source_data if d["gps_source"] == "manual")
n_no_gps = sum(1 for d in cross_source_data if d["gps_source"] is None)

print(f"\n{'='*70}")
print(f"📊 GENEL ÖZET")
print(f"{'='*70}")
print(f"   Toplam fotoğraf  : {len(cross_source_data)}")
print(f"   EXIF GPS'li      : {n_exif}")
print(f"   Manuel GPS'li    : {n_manual}")
print(f"   GPS'siz          : {n_no_gps}")
print(f"   DB-eşleşen klasör: {len(folder_summary) - len(unmatched_folders)} / {len(folder_summary)}")

if n_no_gps > 0:
    print(f"\n⚠️ GPS'i olmayan fotoğraflar (mesafe hatası hesaplanamaz):")
    for d in cross_source_data:
        if d["gps_source"] is None:
            print(f"   - [{d['folder_name']}] {d['filename']}")

if n_exif + n_manual > 0:
    coords = [(d["lat"], d["lon"]) for d in cross_source_data if d["lat"]]
    lats, lons = zip(*coords)
    print(f"\n🗺️ GPS Coğrafi Yayılım:")
    print(f"   Enlem  : {min(lats):.5f} – {max(lats):.5f}")
    print(f"   Boylam : {min(lons):.5f} – {max(lons):.5f}")


In [ ]:
# ============================================================
# Hücre 4: Cross-Source Test — DINOv2 + LightGlue Pipeline
# ============================================================
# Tüm fotoğrafları sırayla test eder. Çok fotoğraf için optimize edildi:
# - tqdm progress bar
# - Detaylı çıktı yerine kompakt log (sadece her 5 fotoğrafta detay)
# - GPU memory yönetimi sıkılaştırıldı
# - Top-5 aday kaydediliyor (Top-3 yerine)

print(f"{'='*65}")
print(f"🔍 CROSS-SOURCE TEST — {len(cross_source_data)} fotoğraf")
print(f"{'='*65}\n")

results = []
VERBOSE_EVERY = 5  # Her 5 fotoğrafta detaylı yazdır

for idx, item in enumerate(tqdm(cross_source_data, desc="Test ediliyor")):
    fname = item["filename"]
    fpath = item["filepath"]
    gt_lat, gt_lon = item["lat"], item["lon"]
    has_gt = gt_lat is not None
    verbose = (idx % VERBOSE_EVERY == 0) or (idx == len(cross_source_data)-1)

    # 1. DINOv2 Embedding
    try:
        img_pil = open_rgb_image(fpath)
        img_tensor = phone_query_transform(img_pil).unsqueeze(0).to(DEVICE)
    except Exception as e:
        print(f"  ❌ {fname}: dosya açılamadı — {e}")
        continue

    t0 = time.time()
    with torch.no_grad():
        query_emb = model(img_tensor).cpu().numpy().astype(np.float32)
    faiss.normalize_L2(query_emb)
    t_dino = time.time() - t0

    # 2. FAISS Top-K
    dists, idxs = faiss_index.search(query_emb, RERANK_TOP_K)
    top_indices = idxs[0]
    top_scores = dists[0]

    dino_best = db_meta[top_indices[0]]
    dino_score = float(top_scores[0])
    dino_err = haversine(gt_lat, gt_lon, dino_best["lat"], dino_best["lon"]) if has_gt else None

    # 3. LightGlue Re-Ranking
    t0 = time.time()
    lg_candidates = []
    img0, feats0 = None, None
    try:
        img0 = load_image(fpath).to(DEVICE)
        with torch.inference_mode():
            feats0 = lg_extractor.extract(img0)

        for k, db_idx in enumerate(top_indices):
            cand = db_meta[db_idx]
            num_matches = 0
            try:
                img1 = load_image(cand["filepath"]).to(DEVICE)
                with torch.inference_mode():
                    feats1 = lg_extractor.extract(img1)
                    matches01 = lg_matcher({"image0": feats0, "image1": feats1})
                matches_clean = rbd(matches01)
                num_matches = int(matches_clean['matches'].shape[0])
                del img1, feats1, matches01, matches_clean
            except Exception as e:
                num_matches = 0
                if verbose and k == 0:
                    print(f"  ⚠️ LightGlue aday hatası ({fname}, Top-1): {type(e).__name__}: {e}")
            lg_candidates.append({
                **cand,
                "num_matches": num_matches,
                "dino_score": float(top_scores[k]),
                "orig_rank": k+1,
                "db_idx": int(db_idx),
            })
    except Exception as e:
        if verbose:
            print(f"  ⚠️ LightGlue toptan hata ({fname}): {e}")
    finally:
        if img0 is not None: del img0
        if feats0 is not None: del feats0
        torch.cuda.empty_cache()

    # En çok eşleşmeye göre sırala
    if lg_candidates:
        lg_candidates.sort(key=lambda x: x["num_matches"], reverse=True)
        lg_best = lg_candidates[0]
    else:
        lg_best = {**dino_best, "num_matches": 0, "orig_rank": 1, "db_idx": int(top_indices[0])}
    t_lg = time.time() - t0
    lg_err = haversine(gt_lat, gt_lon, lg_best["lat"], lg_best["lon"]) if has_gt else None

    # Detaylı log (her N fotoğrafta bir)
    if verbose:
        print(f"\n[{idx+1}/{len(cross_source_data)}] {fname}")
        print(f"  DINOv2 Top-1: {dino_best['street'][:30]} (skor: {dino_score:.3f})"
              + (f", hata: {dino_err:.0f}m" if dino_err is not None else ""))
        print(f"  LightGlue   : {lg_best['street'][:30]} "
              f"({lg_best['num_matches']} eşl., eski sıra: {lg_best['orig_rank']})"
              + (f", hata: {lg_err:.0f}m" if lg_err is not None else ""))

    # Top-5 adayların özetini sakla
    top5_streets = [db_meta[int(top_indices[k])]["street"] for k in range(min(5, len(top_indices)))]
    top5_dino_scores = [float(top_scores[k]) for k in range(min(5, len(top_scores)))]

    results.append({
        "filename": fname,
        "filepath": fpath,
        "folder_street": item.get("folder_street", "?"),
        "gt_lat": gt_lat, "gt_lon": gt_lon,
        "has_gt": has_gt,
        "gps_source": item["gps_source"],
        # DINOv2
        "dino_street": dino_best["street"],
        "dino_lat": dino_best["lat"], "dino_lon": dino_best["lon"],
        "dino_filepath": dino_best["filepath"],
        "dino_score": dino_score,
        "dino_error_m": dino_err,
        "dino_time_ms": t_dino * 1000,
        # LightGlue
        "lg_street": lg_best["street"],
        "lg_lat": lg_best["lat"], "lg_lon": lg_best["lon"],
        "lg_filepath": lg_best["filepath"],
        "lg_matches": lg_best["num_matches"],
        "lg_orig_rank": lg_best["orig_rank"],
        "lg_error_m": lg_err,
        "lg_time_s": t_lg,
        # Top-5 detay
        "top5_streets": top5_streets,
        "top5_dino_scores": top5_dino_scores,
        "lg_candidates": [(c["street"], c["num_matches"], c["orig_rank"]) for c in lg_candidates[:5]],
    })

print(f"\n✅ Test tamamlandı: {len(results)} fotoğraf işlendi.")
if results:
    zero_lg = sum(1 for r in results if r["lg_matches"] == 0)
    print(f"ℹ️ LightGlue 0-eşleşme sayısı: {zero_lg}/{len(results)}")
    if zero_lg == len(results):
        print("⚠️ Tüm LightGlue eşleşmeleri 0 çıktı; sonuçlar DINOv2 ile aynı kalır. Görsel çözünürlük/FOV/domain gap ayrıca incelenmeli.")

In [ ]:
# ============================================================
# Hücre 5: Sonuç Tablosu ve İstatistikler
# ============================================================

df_results = pd.DataFrame(results)
df_with_gt = df_results[df_results["has_gt"]].copy()
df_no_gt = df_results[~df_results["has_gt"]].copy()

print(f"{'='*95}")
print(f"📊 CROSS-SOURCE TEST SONUÇ TABLOSU")
print(f"{'='*95}")
print(f"Toplam test edilen: {len(df_results)} | GPS karşılaştırmalı: {len(df_with_gt)} | GPS'siz: {len(df_no_gt)}")

if len(df_with_gt) > 0:
    # Tüm sonuçlar (kompakt tablo)
    print(f"\n── TÜM SONUÇLAR ({len(df_with_gt)} fotoğraf, GPS hata sırasına göre artan) ──\n")
    df_sorted = df_with_gt.sort_values("lg_error_m")
    print(f"{'#':>3}  {'Dosya':<16} {'GT Sokak':<22} {'DINO Sokak':<22} {'D.Hata':>7} "
          f"{'LG Sokak':<22} {'LG Hata':>7} {'LG#':>4}")
    print("─"*112)
    for i, (_, r) in enumerate(df_sorted.iterrows(), 1):
        d_err = f"{r['dino_error_m']:.0f}m"
        l_err = f"{r['lg_error_m']:.0f}m"
        gt = r.get('folder_street', '?')
        print(f"{i:>3}. {r['filename'][:14]:<16} {gt[:20]:<22} {r['dino_street'][:20]:<22} {d_err:>7} "
              f"{r['lg_street'][:20]:<22} {l_err:>7} {int(r['lg_matches']):>4}")

    # Özet istatistikler
    dino_errors = df_with_gt["dino_error_m"].dropna().values
    lg_errors = df_with_gt["lg_error_m"].dropna().values

    print(f"\n{'='*60}")
    print(f"📈 ÖZET İSTATİSTİKLER")
    print(f"{'='*60}")
    print(f"{'':<28} {'DINOv2':>14} {'LightGlue':>14}")
    print(f"{'─'*60}")
    print(f"{'Medyan Hata (m)':<28} {np.median(dino_errors):>13.1f}  {np.median(lg_errors):>13.1f}")
    print(f"{'Ortalama Hata (m)':<28} {np.mean(dino_errors):>13.1f}  {np.mean(lg_errors):>13.1f}")
    print(f"{'Std (m)':<28} {np.std(dino_errors):>13.1f}  {np.std(lg_errors):>13.1f}")
    print(f"{'P25 (m)':<28} {np.percentile(dino_errors,25):>13.1f}  {np.percentile(lg_errors,25):>13.1f}")
    print(f"{'P75 (m)':<28} {np.percentile(dino_errors,75):>13.1f}  {np.percentile(lg_errors,75):>13.1f}")
    print(f"{'Max (m)':<28} {np.max(dino_errors):>13.1f}  {np.max(lg_errors):>13.1f}")
    print()
    print(f"{'R@1 < 25m':<28} {(dino_errors<=25).mean()*100:>12.1f}% {(lg_errors<=25).mean()*100:>13.1f}%")
    print(f"{'R@1 < 50m':<28} {(dino_errors<=50).mean()*100:>12.1f}% {(lg_errors<=50).mean()*100:>13.1f}%")
    print(f"{'R@1 < 100m':<28} {(dino_errors<=100).mean()*100:>12.1f}% {(lg_errors<=100).mean()*100:>13.1f}%")
    print(f"{'R@1 < 250m':<28} {(dino_errors<=250).mean()*100:>12.1f}% {(lg_errors<=250).mean()*100:>13.1f}%")

    # Same-source referansla karşılaştırma
    print(f"\n{'='*60}")
    print(f"📋 SAME-SOURCE REFERANS (Street View → Street View)")
    print(f"{'='*60}")
    print(f"   R@1 < 100m  : 81.24%")
    print(f"   Medyan hata : 8.2m")
    print(f"   ─────────────────────")
    print(f"   Cross-source ile aradaki fark = DOMAIN GAP")

# GPS'siz fotoğraflar
if len(df_no_gt) > 0:
    print(f"\n── GPS'siz Fotoğraflar (sadece tahmin, {len(df_no_gt)} adet) ──")
    for _, r in df_no_gt.iterrows():
        print(f"   {r['filename']}: DINOv2→{r['dino_street'][:25]}, LG→{r['lg_street'][:25]} ({int(r['lg_matches'])} eşl.)")

In [ ]:
# ============================================================
# Hücre 6: Sokak Bazlı Analiz (klasör adı = ground-truth)
# ============================================================
# Her sorgunun ground-truth sokağı klasör adından geliyor — bu çok daha
# güvenilir, çünkü EXIF GPS'i ±10-20m hatalı olabilir ama klasör adı kesin.
#
# İki ayrı metrik raporlanır:
#   1. Mesafe hatası (haversine) — GPS'li fotoğraflar için
#   2. Sokak adı doğruluğu — TÜM fotoğraflar için (klasör adı vs tahmin)

if len(df_results) > 0 and "folder_street" in df_results.columns:
    df_with_gt_local = df_results[df_results["has_gt"]].copy()

    print(f"{'='*100}")
    print(f"📍 SOKAK BAZLI CROSS-SOURCE PERFORMANSI")
    print(f"{'='*100}")
    print(f"{'Ground-Truth Sokak (klasör)':<32} {'N':>3} {'NGps':>5} {'Med.D.Hata':>11} "
          f"{'Med.LG.Hata':>12} {'<100m':>7} {'DINO Sok.✓':>11} {'LG Sok.✓':>10}")
    print("─"*100)

    rows = []
    for gt_street, group_all in df_results.groupby("folder_street"):
        n = len(group_all)
        # GPS'li alt küme
        group_gps = group_all[group_all["has_gt"]]
        n_gps = len(group_gps)
        if n_gps > 0:
            med_d = group_gps["dino_error_m"].median()
            med_lg = group_gps["lg_error_m"].median()
            r100 = (group_gps["lg_error_m"] <= 100).mean() * 100
        else:
            med_d, med_lg, r100 = None, None, None

        # Sokak adı doğruluğu — TÜM fotoğraflar için
        dino_correct = group_all.apply(lambda r: norm_text(r["dino_street"]) == norm_text(gt_street), axis=1).mean() * 100
        lg_correct = group_all.apply(lambda r: norm_text(r["lg_street"]) == norm_text(gt_street), axis=1).mean() * 100

        rows.append((gt_street, n, n_gps, med_d, med_lg, r100, dino_correct, lg_correct))

    rows.sort(key=lambda x: x[4] if x[4] is not None else float('inf'))

    for gt_street, n, n_gps, med_d, med_lg, r100, dino_c, lg_c in rows:
        med_d_str = f"{med_d:>10.0f}m" if med_d is not None else f"{'-':>11}"
        med_lg_str = f"{med_lg:>11.0f}m" if med_lg is not None else f"{'-':>12}"
        r100_str = f"{r100:>6.0f}%" if r100 is not None else f"{'-':>7}"
        print(f"{gt_street[:30]:<32} {n:>3} {n_gps:>5} {med_d_str} {med_lg_str} "
              f"{r100_str} {dino_c:>10.0f}% {lg_c:>9.0f}%")

    # Genel sokak adı doğruluğu
    overall_dino = df_results.apply(lambda r: norm_text(r["dino_street"]) == norm_text(r["folder_street"]), axis=1).mean() * 100
    overall_lg = df_results.apply(lambda r: norm_text(r["lg_street"]) == norm_text(r["folder_street"]), axis=1).mean() * 100

    print(f"\n{'='*60}")
    print(f"🎯 GENEL SOKAK ADI DOĞRULUĞU")
    print(f"{'='*60}")
    print(f"   DINOv2 Top-1 doğru sokak    : {overall_dino:.1f}%")
    print(f"   LightGlue Top-1 doğru sokak : {overall_lg:.1f}%")
    print(f"\n💡 Sütun açıklamaları:")
    print(f"   N         = toplam fotoğraf sayısı (klasördeki)")
    print(f"   NGps      = GPS bilgisi olan fotoğraf sayısı")
    print(f"   Med.D.Hata = DINOv2 medyan mesafe hatası (sadece GPS'li)")
    print(f"   Med.LG.Hata = LightGlue medyan mesafe hatası")
    print(f"   <100m     = LightGlue 100m altı hata oranı")
    print(f"   DINO/LG Sok.✓ = doğru SOKAĞI tahmin etme oranı")
    print(f"\n   Düşük 'Sok.✓' ama düşük 'Med.Hata' = komşu sokağa atıyor (yakın yanlış)")
    print(f"   Yüksek 'Sok.✓' = aliasing sorunu yok, model sokağı doğru tanıyor")


In [ ]:
# ============================================================
# Hücre 7: Akıllı Görselleştirme — Sadece İlginç Vakalar
# ============================================================
# Tüm fotoğrafları görselleştirmek yerine: en iyi 3, en kötü 3,
# ve eğer varsa 'aliasing üçgeni' fotoğraflarını göster.

def find_db_record_by_filepath(filepath):
    """db_meta içinde tam filepath eşleşmesi yapar."""
    for m in db_meta:
        if m["filepath"] == filepath:
            return m
    return None

def safe_open_image(filepath):
    """Dosya yoksa placeholder yerine None döner."""
    try:
        return open_rgb_image(filepath)
    except Exception:
        return None

def plot_one_query(res, save_dir=None):
    """Bir sorgu için 4'lü panel: sorgu + DINO Top-1 + LG Top-1 + Top-2."""
    fig, axes = plt.subplots(1, 4, figsize=(18, 4))

    # 1. Sorgu
    q_img = safe_open_image(res["filepath"])
    if q_img:
        axes[0].imshow(q_img)
    title_q = f"SORGU [{res.get('folder_street', '?')[:20]}]\n{res['filename'][:25]}"
    if res["has_gt"]:
        title_q += f"\nGT: {res['gt_lat']:.4f}, {res['gt_lon']:.4f}"
    axes[0].set_title(title_q, fontsize=9, color="blue", fontweight="bold")
    axes[0].axis("off")

    # 2. DINOv2 Top-1
    d_img = safe_open_image(res["dino_filepath"])
    if d_img:
        axes[1].imshow(d_img)
    color = "green" if (res["dino_error_m"] is not None and res["dino_error_m"] < 100) else "red"
    d_title = f"DINOv2 Top-1\n{res['dino_street'][:25]}\nSkor: {res['dino_score']:.3f}"
    if res["dino_error_m"] is not None:
        d_title += f"\nHata: {res['dino_error_m']:.0f}m"
    axes[1].set_title(d_title, fontsize=9, color=color, fontweight="bold")
    axes[1].axis("off")

    # 3. LightGlue Top-1
    l_img = safe_open_image(res["lg_filepath"])
    if l_img:
        axes[2].imshow(l_img)
    color = "green" if (res["lg_error_m"] is not None and res["lg_error_m"] < 100) else "red"
    l_title = f"LightGlue Top-1\n{res['lg_street'][:25]}\n{res['lg_matches']} eşleşme"
    if res["lg_error_m"] is not None:
        l_title += f"\nHata: {res['lg_error_m']:.0f}m"
    axes[2].set_title(l_title, fontsize=9, color=color, fontweight="bold")
    axes[2].axis("off")

    # 4. DINOv2 Top-2 (referans için)
    if len(res["top5_streets"]) > 1:
        # Top-2'nin filepath'ini bulmak için orijinal arama yapmak gerekir.
        # Onu kaydetmedik; fakat sokak ismi + skor yeterince bilgilendirici.
        axes[3].text(0.5, 0.5,
                     f"DINOv2 Top-2:\n{res['top5_streets'][1]}\n\n"
                     f"Top-3:\n{res['top5_streets'][2] if len(res['top5_streets'])>2 else '-'}\n\n"
                     f"Top-4:\n{res['top5_streets'][3] if len(res['top5_streets'])>3 else '-'}",
                     ha="center", va="center", fontsize=10,
                     transform=axes[3].transAxes)
    axes[3].set_title("Diğer Adaylar", fontsize=9)
    axes[3].axis("off")

    plt.tight_layout()

    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
        out_path = os.path.join(save_dir, f"vis_{res['filename'].rsplit('.',1)[0]}.png")
        plt.savefig(out_path, dpi=110, bbox_inches="tight")
    plt.show()

# ─── Hangi vakaları göstereceğimizi seç ───
vis_save_dir = os.path.join(RESULTS_DIR, "cross_source_visuals")

if len(df_with_gt) > 0:
    df_sorted = df_with_gt.sort_values("lg_error_m")
    # En iyi 3 (en düşük hata)
    best_3 = df_sorted.head(3)
    # En kötü 3 (en yüksek hata)
    worst_3 = df_sorted.tail(3).iloc[::-1]

    print(f"{'='*65}")
    print(f"🏆 EN İYİ 3 SONUÇ (en düşük LightGlue hatası)")
    print(f"{'='*65}")
    for _, r in best_3.iterrows():
        plot_one_query(r.to_dict(), save_dir=vis_save_dir)

    print(f"\n{'='*65}")
    print(f"💥 EN KÖTÜ 3 SONUÇ (en yüksek LightGlue hatası)")
    print(f"{'='*65}")
    for _, r in worst_3.iterrows():
        plot_one_query(r.to_dict(), save_dir=vis_save_dir)

    print(f"\n💾 Tüm görseller {vis_save_dir} klasörüne kaydedildi.")
else:
    # GPS yoksa tüm tahminleri göster (max 10)
    print("GPS'siz fotoğraflar — sadece ilk 10 tahmin gösteriliyor:")
    for _, r in df_results.head(10).iterrows():
        plot_one_query(r.to_dict(), save_dir=vis_save_dir)

In [ ]:
# ============================================================
# Hücre 8: Toplu Folium Haritası — Tüm sorgular tek haritada
# ============================================================
# Her sorgu için ayrı harita yerine tek bir interaktif harita.

if len(df_with_gt) > 0:
    center_lat = df_with_gt["gt_lat"].mean()
    center_lon = df_with_gt["gt_lon"].mean()

    m_map = folium.Map(location=[center_lat, center_lon], zoom_start=14, tiles="OpenStreetMap")

    for _, r in df_with_gt.iterrows():
        # Hata seviyesine göre renk
        if r["lg_error_m"] is None:
            color = "gray"
        elif r["lg_error_m"] < 50:
            color = "green"
        elif r["lg_error_m"] < 250:
            color = "orange"
        else:
            color = "red"

        # GT marker
        folium.CircleMarker(
            [r["gt_lat"], r["gt_lon"]],
            radius=5, color="blue", fill=True, fill_opacity=0.8,
            popup=folium.Popup(
                f"<b>SORGU: {r['filename']}</b><br>"
                f"GT: {r['gt_lat']:.5f}, {r['gt_lon']:.5f}<br><br>"
                f"<b>DINOv2:</b> {r['dino_street']}<br>"
                f"  Skor: {r['dino_score']:.3f}<br>"
                f"  Hata: {r['dino_error_m']:.0f}m<br><br>"
                f"<b>LightGlue:</b> {r['lg_street']}<br>"
                f"  {r['lg_matches']} eşleşme (eski sıra: {r['lg_orig_rank']})<br>"
                f"  Hata: {r['lg_error_m']:.0f}m",
                max_width=300)
        ).add_to(m_map)

        # Hata çizgisi
        folium.PolyLine(
            [[r["gt_lat"], r["gt_lon"]], [r["lg_lat"], r["lg_lon"]]],
            color=color, weight=2, opacity=0.7, dash_array="5"
        ).add_to(m_map)

        # LG tahmin marker'ı
        folium.CircleMarker(
            [r["lg_lat"], r["lg_lon"]],
            radius=3, color=color, fill=True, fill_opacity=0.6
        ).add_to(m_map)

    # Lejant
    legend_html = '''
    <div style="position: fixed; top: 10px; right: 10px; background: white;
                padding: 10px; border: 2px solid grey; z-index: 9999; font-size: 12px;">
      <b>Cross-Source Sonuçları</b><br>
      🔵 Sorgu (GT) konumu<br>
      🟢 < 50m hata<br>
      🟠 50–250m hata<br>
      🔴 > 250m hata
    </div>
    '''
    m_map.get_root().html.add_child(folium.Element(legend_html))

    # HTML olarak kaydet (büyük çıktı browser'ı kasmasın)
    map_path = os.path.join(RESULTS_DIR, "cross_source_map.html")
    m_map.save(map_path)
    print(f"✅ İnteraktif harita kaydedildi: {map_path}")
    print(f"   Drive'dan açıp tarayıcıda görüntüleyebilirsin.")
    display(m_map)

In [ ]:
# ============================================================
# Hücre 9: Sonuçları Kaydet ve Makale Özeti
# ============================================================

# CSV (ana tablo, list-tipindeki kolonlar string'e dönüştürülür)
df_save = df_results.copy()
df_save["top5_streets"] = df_save["top5_streets"].apply(lambda x: " | ".join(map(str, x)))
df_save["top5_dino_scores"] = df_save["top5_dino_scores"].apply(
    lambda x: " | ".join(f"{s:.3f}" for s in x))
df_save["lg_candidates"] = df_save["lg_candidates"].apply(
    lambda lst: " | ".join(f"{s}({n}eşl,sıra{r})" for s,n,r in lst))

csv_path = os.path.join(RESULTS_DIR, "cross_source_results.csv")
df_save.to_csv(csv_path, index=False, encoding="utf-8-sig")
print(f"✅ Tam sonuç tablosu kaydedildi: {csv_path}")

# Makale özeti
dino_street_acc = df_results.apply(lambda r: norm_text(r["dino_street"]) == norm_text(r["folder_street"]), axis=1).mean() * 100 if len(df_results) else 0
lg_street_acc = df_results.apply(lambda r: norm_text(r["lg_street"]) == norm_text(r["folder_street"]), axis=1).mean() * 100 if len(df_results) else 0
lg_zero_matches = int((df_results["lg_matches"] == 0).sum()) if len(df_results) else 0
summary_path = os.path.join(RESULTS_DIR, "cross_source_summary.txt")
with open(summary_path, "w", encoding="utf-8") as f:
    f.write("="*65 + "\n")
    f.write("CROSS-SOURCE TEST ÖZETİ — Makale için\n")
    f.write("="*65 + "\n\n")
    f.write(f"Test edilen fotoğraf sayısı : {len(results)}\n")
    f.write(f"GPS karşılaştırmalı        : {len(df_with_gt)}\n")
    f.write(f"GPS'siz                    : {len(df_no_gt)}\n")
    f.write(f"Kaynak                     : Cep telefonu kamerası\n")
    f.write(f"Referans veritabanı        : Google Street View\n")
    f.write(f"Veritabanı boyutu          : {faiss_index.ntotal} görüntü, "
            f"{len(set(m['street'] for m in db_meta))} sokak\n")
    f.write(f"DINOv2 sokak doğruluğu     : {dino_street_acc:.1f}%\n")
    f.write(f"LightGlue sokak doğruluğu  : {lg_street_acc:.1f}%\n")
    f.write(f"LightGlue 0-eşleşme        : {lg_zero_matches}/{len(df_results)}\n\n")

    if len(df_with_gt) > 0:
        dino_errors = df_with_gt["dino_error_m"].dropna().values
        lg_errors = df_with_gt["lg_error_m"].dropna().values

        f.write("DINOv2 Sonuçları (cross-source):\n")
        f.write(f"  Medyan hata  : {np.median(dino_errors):.1f}m\n")
        f.write(f"  Ortalama hata: {np.mean(dino_errors):.1f}m\n")
        f.write(f"  R@1 < 25m    : {(dino_errors<=25).mean()*100:.1f}%\n")
        f.write(f"  R@1 < 100m   : {(dino_errors<=100).mean()*100:.1f}%\n\n")

        f.write("DINOv2 + LightGlue Sonuçları (cross-source):\n")
        f.write(f"  Medyan hata  : {np.median(lg_errors):.1f}m\n")
        f.write(f"  Ortalama hata: {np.mean(lg_errors):.1f}m\n")
        f.write(f"  R@1 < 25m    : {(lg_errors<=25).mean()*100:.1f}%\n")
        f.write(f"  R@1 < 100m   : {(lg_errors<=100).mean()*100:.1f}%\n\n")

        f.write("Same-source referans (Street View → Street View):\n")
        f.write(f"  R@1 < 100m   : 81.24%\n")
        f.write(f"  Medyan hata  : 8.2m\n\n")

        f.write("Domain gap = same-source ile cross-source arası fark.\n")

print(f"✅ Makale özeti kaydedildi: {summary_path}")
print()
print("="*65)
print("✅ Notebook 4 v2 tamamlandı.")
print("="*65)
print()
print("📁 Çıktılar:")
print(f"   • {csv_path}")
print(f"   • {summary_path}")
print(f"   • {os.path.join(RESULTS_DIR, 'cross_source_map.html')}")
print(f"   • {os.path.join(RESULTS_DIR, 'cross_source_visuals/')} (en iyi/kötü 3 görsel)")